# CNNs for Vision: ResNet, EfficientNet, ViT & Interpretability

**Module 4 - Notebook 2** | Skillsoft Track 2: DL Programmer

## Objectives
- Train ResNet/EfficientNet on CIFAR-10
- Vision Transformer (ViT) implementation
- Grad-CAM visualization
- Detection/segmentation overview

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import timm
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# QUICK_MODE keeps CI execution fast (subset + 1 epoch)
import os
QUICK_MODE = os.getenv("QUICK_MODE", "0") == "1"
EPOCHS = 1 if QUICK_MODE else 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")

## 1. CIFAR-10 Data Loading

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

if QUICK_MODE:
    # Offline synthetic CIFAR-shaped data: no download, fast CI execution
    train_dataset = torchvision.datasets.FakeData(
        size=2000, image_size=(3, 32, 32), num_classes=10, transform=transforms.ToTensor())
    test_dataset = torchvision.datasets.FakeData(
        size=500, image_size=(3, 32, 32), num_classes=10, transform=transforms.ToTensor())
else:
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

workers = 0 if QUICK_MODE else 2
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=workers)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=workers)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

## 2. ResNet-18 with TIMM

In [ ]:
# ResNet-18 from TIMM
model = timm.create_model('resnet18', pretrained=False, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(loader, desc='Training'):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

def eval_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

# Quick training (5 epochs for demo)
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = eval_model(model, test_loader, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Acc={train_acc:.2f}% | Test Loss={test_loss:.4f}, Acc={test_acc:.2f}%")

## 3. EfficientNet-B0

In [ ]:
# EfficientNet-B0
effnet = timm.create_model('efficientnet_b0', pretrained=False, num_classes=10).to(device)

optimizer_eff = torch.optim.AdamW(effnet.parameters(), lr=3e-4, weight_decay=0.05)
scheduler_eff = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=50)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(effnet, train_loader, criterion, optimizer_eff, device)
    test_loss, test_acc = eval_model(effnet, test_loader, criterion, device)
    scheduler_eff.step()
    print(f"Epoch {epoch}: Train Acc={train_acc:.2f}% | Test Acc={test_acc:.2f}%")

## 4. Vision Transformer (ViT) from Scratch

In [ ]:
# Simple ViT implementation
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, n_patches, embed_dim)
        return x

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 embed_dim=192, depth=6, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=int(embed_dim * mlp_ratio),
                dropout=dropout,
                activation='gelu',
                batch_first=True,
            )
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        x = x[:, 0]  # CLS token
        x = self.head(x)
        return x

vit = ViT().to(device)
print(f"ViT params: {sum(p.numel() for p in vit.parameters())/1e6:.2f}M")

In [ ]:
# Train ViT (few epochs)
optimizer_vit = torch.optim.AdamW(vit.parameters(), lr=1e-3, weight_decay=0.05)
scheduler_vit = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_vit, T_max=50)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(vit, train_loader, criterion, optimizer_vit, device)
    test_loss, test_acc = eval_model(vit, test_loader, criterion, device)
    scheduler_vit.step()
    print(f"Epoch {epoch}: Train Acc={train_acc:.2f}% | Test Acc={test_acc:.2f}%")

## 5. Grad-CAM Visualization

In [ ]:
# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        self.activations = output
    
    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
    
    def __call__(self, x, class_idx=None):
        self.model.eval()
        output = self.model(x)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        self.model.zero_grad()
        output[0, class_idx].backward()
        
        # Global average pooling of gradients
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        return cam.squeeze().cpu().detach().numpy(), class_idx

# Apply to ResNet
gradcam = GradCAM(model, model.layer4[-1].conv2)

# Get sample image
sample_img, sample_label = test_dataset[0]
sample_img = sample_img.unsqueeze(0).to(device)

cam, pred_class = gradcam(sample_img)

# Visualize
img_np = sample_img.squeeze().cpu().numpy().transpose(1, 2, 0)
img_np = img_np * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
img_np = np.clip(img_np, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_np)
axes[0].set_title(f'Original: {classes[sample_label]}')
axes[0].axis('off')

axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM')
axes[1].axis('off')

axes[2].imshow(img_np)
axes[2].imshow(cam, cmap='jet', alpha=0.5)
axes[2].set_title(f'Overlay (Pred: {classes[pred_class]})')
axes[2].axis('off')

plt.tight_layout()
plt.show()